# Step 1: Data Exploration & Cleaning

For the Tardis project , we will need some libraries

In [ ]:
import pandas as pd # panda for open a csv and extract the data for file and data manipulation
import matplotlib as pl # matplotlib to create visualisation via graphs

This are some macros to change the way the code is 

In [ ]:
# this is the filename use for the analisys
DATASET = "dataset.csv" 

# this the list of the columns name that have to be transform to numeric
COLUMNS_TO_NUMERIC = ['Average journey time', 'Number of scheduled trains', 'Number of cancelled trains', 'Number of trains delayed at departure', 
                      'Average delay of late trains at departure', 'Average delay of all trains at departure', 'Number of trains delayed at arrival',
                      'Average delay of late trains at arrival', 'Average delay of all trains at arrival', 'Number of trains delayed > 15min', 
                      'Average delay of trains > 15min (if competing with flights)', 'Number of trains delayed > 30min',
                      'Number of trains delayed > 60min', 'Pct delay due to external causes', 'Pct delay due to infrastructure',
                      'Pct delay due to traffic management', 'Pct delay due to rolling stock', 
                      'Pct delay due to station management and equipment reuse', 
                      'Pct delay due to passenger handling (crowding, disabled persons, connections)']

# this is the column name for departure station
DEPARTURE = 'Departure station'

# this is the column name for arrival station
ARRIVAL = 'Arrival station'

# create a type reference dataframe 
type dataframe = pd.DataFrame


## Open the dataset and clean it

Open the file and save the dataframe

In [ ]:
raw_df = pd.read_csv(DATASET, on_bad_lines="skip", sep=";")

And this is a tab representation of the file

In [ ]:
raw_df

## Let's clean our dataframe

We choose to remove the whole row if there are duplicates only and keep the line with empty cell.

Now we remove duplicate rows

In [ ]:
df = raw_df.drop_duplicates(ignore_index=True) # df is the result of the action of removing rows
# ignore_index=True is a parameter that permit to give new index to the df's rows, without it the new rows will keep their old index

The duplicates rows are now removed. For the empty cell we choose to fill them depending of the other values. But first we need to transform some columns into numeric values.

In [ ]:
for col in COLUMNS_TO_NUMERIC: # COLUMNS_TO_NUMERIC is a list of the columns we have to change
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '.'), errors="coerce") # we transform "str" value to numeric
# and we replace the dot by coma for decimal values

We can change the format for the date

In [ ]:
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m", errors="coerce").dt.to_period("M")
# the function above transform the "str" value to a time value as we figured out that the format used in the file was year-month
# so we are keeping the year and the month only and if the value can't be tranform "Nat" will be display instead

We need to standardise the station name: no caps, no space at the beginning and weed need a value.

This is the standardise station function, we simply call the function with the dataframe and the column to standardise and Voilà!

In [ ]:
def standardise_station(df:dataframe, column_name:str) -> None:
    """
    Standardise the column value in the dataframe

    Parameters
    ----------
    df : pd.DataFrame
        a dataframe.
    column_name : str
        the column name.

    Returns
    -------
    None
    """
    df[column_name] = df[column_name].str.strip() # remove space at start and end
    df[column_name] = df[column_name].str.lower() # lower characters
    df[column_name] = df[column_name].replace({r"\bst\b":"saint"}, regex=True) # change "st" to "saint"

# strip function will remove spaces at the start and at the end of the string
# lower function will change all characters to their lower version
# we are using strip and not lstrip (remove the space at the start) nor rstrip (remove the space at the start)
# replace function will take any "st" and transform it to "saint" , \b will delimit the partern "st", a 2 letter word and not a part of a word
# to use the \b we need to use a raw string ( r"....." ) and we need to says regex is true otherwise that will don't work the way we want 

In [ ]:
standardise_station(df, DEPARTURE) # standardise the departure station
standardise_station(df, ARRIVAL) # standardise the arrival station
print(df[DEPARTURE])

## Fill the gap

Now we transform our values in a type we want, we can use the values to fill empty cell. We need to fill the empty numercial values. We have the choice between three function: ffill , bfill and interpolate.

### What function to use?

"ffill" function will take the forward value and copy paste it to the empty cells.
By the same principle "bfill" function will take the backward value and copy paste it to the empty cells.
Fortunatly there is the "interpolate" function that will do calculation and give us a value depending of the other values of the dataframe. 
This is not perfect, but it's far better that simply copy pasting a neighbour value.

In [ ]:
df[COLUMNS_TO_NUMERIC] = df[COLUMNS_TO_NUMERIC].interpolate()
df

Get all station name

In [ ]:
df_departure_station = df.dropna(subset=['Departure station'])
stations = df_departure_station['Departure station'].unique()
for station in stations:
    print(station)